# Lesson 1 · What is retrieval-augmented detection?

<a href="https://colab.research.google.com/github/Mary-Vadi/rag/blob/main/lessons/01_what_is_retrieval_augmented_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**In this lesson you will**

- understand the task: deciding whether a text was written by a person or generated by an AI model
- see the two ways a detector can make that decision, and how this course combines them
- find out how our method relates to "RAG" as it's used for chatbots
- load the two datasets used throughout the course and look at real examples

**You need:** basic Python. No machine-learning background is assumed.
**Runtime:** a CPU is enough for this lesson.

## 1. The task

We're given a piece of text and have to answer one question: **was this written by a human or generated by a machine?**

We'll write the answer as a number, the *label*:

| Label | Meaning |
|---|---|
| `0` | human-written |
| `1` | machine-generated |

This is called **binary classification**: every text gets exactly one of two labels.

Why is it hard? Modern language models write fluent, grammatical text, so there is no single give-away. A detector has to pick up subtle statistical habits (word choice, rhythm, how predictable each word is), and those habits change with the model, the topic and the prompt. A detector that works well on one kind of text can fail badly on another. That problem, called **domain shift**, is the thread running through this whole course.

## 2. Two ways to make a decision

Imagine you're a teacher asked whether an essay was written by a student or by an AI tool. You could work in one of two ways.

**Option A: trust your gut (a classifier).** You've read thousands of essays of both kinds, and over time you've built up an instinct. You read the new essay and decide on the spot. This is fast, but your instinct only covers the kinds of essay you practised on. Hand you a poem or a recipe and it may be useless.

**Option B: look it up (retrieval).** You keep a filing cabinet of essays that are already labelled. When a new essay arrives, you find the handful that are *most similar* to it and check how they were labelled. If most of them were AI-written, you guess AI. You need no training, and you can add new examples to the cabinet at any time. But you're only as good as your cabinet, and your sense of what "similar" means.

**Retrieval-augmented detection** does both and combines the answers:

```
                  ┌───────────────────────────┐
  new text ──┬──▶ │ classifier                │ ── P(machine) ──┐
             │    └───────────────────────────┘                 ▼
             │    ┌───────────────────────────┐            ┌─────────┐
             └──▶ │ find the most similar     │ ── vote ──▶│  blend  │──▶ human or machine
                  │ texts in a labelled pool  │            └─────────┘
                  └───────────────────────────┘
```

Each side produces a probability that the text is machine-generated, `P(machine)`, a number between 0 and 1. The blend is a weighted average of the two, and anything at 0.5 or above counts as "machine".

## 3. Is this "RAG"?

You've probably heard of **RAG**, short for *Retrieval-Augmented **Generation***. It's how many chatbots answer questions about documents:

```
question ──▶ retrieve relevant documents ──▶ an LLM reads them and writes an answer
```

Our method keeps the first half, retrieval, but replaces the generation step with a decision. **There is no large language model (LLM) involved.**

| | RAG for chatbots | Retrieval-augmented detection (this course) |
|---|---|---|
| What is retrieved? | documents that help answer the question | labelled texts similar to the one being judged |
| What happens next? | an LLM writes an answer from them | their labels are turned into a vote |
| Output | generated text | human or machine |
| Needs an LLM? | yes | no |

The good news is that the core skills are the same. Turning text into vectors (**embeddings**, lesson 2) and finding similar vectors quickly (**similarity search**, lesson 3) are exactly what every RAG system is built on. Everything you learn here transfers directly.

## 4. The course at a glance

| Lesson | What you build | Runtime |
|---|---|---|
| **1. What is retrieval-augmented detection?** | load and explore the data | CPU |
| **2. Embeddings** | turn texts into vectors and compare them | CPU or GPU |
| **3. Similarity search** | find a text's nearest neighbours with FAISS | CPU or GPU |
| **4. Retrieval as a detector** | a detector that only uses neighbours, and how to evaluate it | CPU or GPU |
| **5. Training a classifier** | fine-tune ModernBERT, and watch it struggle on new data | GPU |
| **6. Retrieval-augmented detection** | combine the two and tune the blend | GPU |

The code you write in each lesson also lives in the `ragdetect/` folder of this repository, so later lessons can reuse it with a single import.

## 5. Setup

Run the next cell before anything else. In Colab it downloads this course's code and installs a few libraries, which takes about a minute the first time.

In [ ]:
# Setup: run this cell first.
# In Colab it downloads the course code and installs the extra libraries.
# On your own computer (from the lessons/ folder) it just makes the code importable.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/rag"):
        !git clone -q https://github.com/Mary-Vadi/rag.git /content/rag
        !pip install -q -r /content/rag/requirements.txt
    %cd /content/rag
elif os.path.basename(os.getcwd()) == "lessons":
    %cd ..
sys.path.insert(0, os.getcwd())

In [ ]:
import textwrap

import matplotlib.pyplot as plt
import numpy as np

from ragdetect.data import LABEL_NAMES, load_mage, load_raid

## 6. Meet the data

We use two public benchmarks. Both are downloaded from the Hugging Face Hub the first time you load them.

- **RAID** ([paper](https://arxiv.org/abs/2405.07940)) has more than 10 million texts from 11 language models, covering news, books, scientific abstracts, reviews, Reddit, recipes, Wikipedia and poetry, many of them deliberately disguised with "adversarial attacks" such as misspellings. We use RAID to **train** the classifier in lesson 5.
- **MAGE** ([paper](https://arxiv.org/abs/2305.13242)) has about 436,000 texts. The human texts come from 10 different sources (Reddit discussions, news, stories, reviews, scientific writing and more), and the machine texts from 27 different language models. We use MAGE to **test** our detectors and as the **retrieval pool**.

The two datasets store their labels differently. MAGE even uses the *opposite* numbers (1 for human). The loaders in `ragdetect/data.py` hide all of that and give back a table with the same three columns every time:

| Column | Contents |
|---|---|
| `text` | the passage |
| `label` | 0 = human, 1 = machine |
| `source` | where it came from, e.g. a model name |

Let's load 2,000 texts from MAGE's test set, half human and half machine.

In [ ]:
mage = load_mage("test", n=2000)
mage.head()

In [ ]:
print(mage["label"].map(dict(enumerate(LABEL_NAMES))).value_counts())

The `source` column tells us where each text came from. Human texts end in `_human`. Machine texts name the original dataset and the model that wrote them.

In [ ]:
mage["source"].value_counts().head(10)

A small helper to print a text with its label makes the examples easier to read:

In [ ]:
def show(df, i, width=500):
    row = df.iloc[i]
    print(f"[{LABEL_NAMES[row.label]} | {row.source}]")
    print(textwrap.fill(textwrap.shorten(row.text, width), 100))
    print()

for i in range(3):
    show(mage, i)

Now RAID. Its training file is 11.8 GB, so instead of downloading all of it, `load_raid` reads the first 100,000 rows as a stream and takes a balanced sample from those. It skips texts that went through an adversarial attack. This takes a minute or two.

In [ ]:
raid = load_raid(n=4000)
print(raid["label"].map(dict(enumerate(LABEL_NAMES))).value_counts(), "\n")
print(raid["source"].value_counts(), "\n")
print(raid["domain"].value_counts())

> **Keep an eye on the `domain` column.** RAID's file is sorted by domain, so every text in our sample comes from the same one. That shortcut will matter a lot in lesson 5.

In [ ]:
for i in range(2):
    show(raid, i)

## 7. Is text length a give-away?

A natural first idea: maybe machines write longer (or shorter) texts than people. Let's compare the number of words in each MAGE text by label.

In [ ]:
words = mage["text"].str.split().str.len()

plt.figure(figsize=(7, 3.5))
for label, name in enumerate(LABEL_NAMES):
    plt.hist(words[mage["label"] == label], bins=40, range=(0, 800), alpha=0.6, label=name)
plt.xlabel("words per text")
plt.ylabel("number of texts")
plt.title("MAGE test sample: text length by label")
plt.legend()
plt.show()

If the two histograms overlap heavily, length alone can't separate them, and we need something that understands *what* the text says and *how*. That's where embeddings come in.

## 8. Your turn: human or machine?

Before any model gets involved, test yourself. The next cell shows six MAGE texts without their labels. Write down your guesses, then run the cell after it.

In [ ]:
quiz = mage.sample(6, random_state=7).reset_index(drop=True)
for i, text in enumerate(quiz["text"]):
    print(f"--- Text {i + 1} ---")
    print(textwrap.fill(textwrap.shorten(text, 400), 100), "\n")

In [ ]:
for i, row in quiz.iterrows():
    print(f"Text {i + 1}: {LABEL_NAMES[row.label]}  ({row.source})")

How did you do? Most people find this surprisingly hard, which is exactly why we need automatic detectors.

## Check your understanding

<details>
<summary>1. Why do our loaders flip MAGE's labels?</summary>

MAGE uses 1 for human and 0 for machine, the opposite of the convention in this course. Mixing up conventions between datasets is one of the most common (and most silent) bugs in machine learning, so we convert everything to one convention as soon as we load it.
</details>

<details>
<summary>2. What is the difference between a classifier and a retrieval-based detector?</summary>

A classifier learns patterns during training and then judges each new text on its own. A retrieval-based detector learns nothing; it looks up the most similar labelled texts and copies their labels. Retrieval can be updated just by adding examples, but it depends entirely on how good the pool and the similarity measure are.
</details>

<details>
<summary>3. Does our method need a large language model?</summary>

No. Classic RAG uses an LLM to *generate* an answer from the retrieved documents. We only use the retrieved texts' labels to vote, so there is nothing to generate.
</details>

## Summary

- The task is binary classification: 0 = human, 1 = machine.
- A classifier decides from what it learned; retrieval decides from similar labelled examples; retrieval-augmented detection blends the two.
- Our method uses the "retrieval" part of RAG without the "generation" part, so it needs no LLM.
- We train on RAID and test on MAGE, which come from quite different kinds of writing.

**Next:** [Lesson 2 · Embeddings](02_embeddings.ipynb), where we turn text into numbers a computer can compare.